# NCR-Match pipeline notebook

Pipeline: `retrieve.py` → `geometry_filter.py` → `vggt_signals.py` → `pose_scoring.py`

**Section A** — Session setup (run once per fresh Colab session)  
**Section B** — Configuration (edit as needed)  
**Section C** — Staging (run when resetting or updating from Drive)  
**Section D** — Pipeline (each step independently re-runnable)  
**Section E** — Inspect

---
## Section A — Session setup
*Run once per fresh Colab session.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, sys
from pathlib import Path


def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)


_pip('torchmetrics', 'albumentations', 'pytorch-lightning')

ASPAN_REQS = Path('/content/drive/MyDrive/ImageSimilarity/ml-aspanformer/requirements.txt')
if ASPAN_REQS.exists():
    _pip('-r', str(ASPAN_REQS))
else:
    print('WARNING: ASpanFormer requirements.txt not found at', ASPAN_REQS)

_pip('git+https://github.com/facebookresearch/vggt-omega.git')
print('All installs done.')

---
## Section B — Configuration
*Edit as needed; re-running has no side effects.*

In [ ]:
from pathlib import Path

# ── Drive paths ───────────────────────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/ImageSimilarity')
LOCAL_ROOT  = Path('/content/image_similarity_dev')

# Archives
SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'

# Pipeline scripts (copied to LOCAL_ROOT during staging)
RETRIEVE_SCRIPT_DRIVE    = DRIVE_ROOT / 'retrieve.py'
GEOFILTER_SCRIPT_DRIVE   = DRIVE_ROOT / 'geometry_filter.py'
VGGTSIGNALS_SCRIPT_DRIVE = DRIVE_ROOT / 'vggt_signals.py'
POSESCORING_SCRIPT_DRIVE = DRIVE_ROOT / 'pose_scoring.py'
MODEL_DRIVE              = DRIVE_ROOT / 'ModelComboDINO.py'
MODEL_CLASS              = 'ModelComboDINO'

# Checkpoints on Drive
DINO_CHECKPOINT_DRIVE    = DRIVE_ROOT / 'DINO.pt'
DINO_V3_CHECKPOINT_NAME  = 'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
DINO_V3_CHECKPOINT_DRIVE = DRIVE_ROOT / DINO_V3_CHECKPOINT_NAME
VGGT_CHECKPOINT_DRIVE    = DRIVE_ROOT / 'weights/VGGT-Omega/vggt_omega_1b_512.pt'
ASPAN_ROOT               = DRIVE_ROOT / 'ml-aspanformer'
ASPAN_CONFIG             = ASPAN_ROOT / 'configs/aspan/outdoor/aspan_test.py'
ASPAN_WEIGHTS_DRIVE      = DRIVE_ROOT / 'aspanweights/outdoor.ckpt'

# ── Local paths (derived) ─────────────────────────────────────────────────────
WEIGHTS_DIR              = LOCAL_ROOT / 'weights'
MATCH_WEIGHTS            = WEIGHTS_DIR / 'best.pt'
DINO_V3_CHECKPOINT_LOCAL = LOCAL_ROOT / DINO_V3_CHECKPOINT_NAME
VGGT_CHECKPOINT_LOCAL    = LOCAL_ROOT / 'vggt_omega_1b_512.pt'
ASPAN_WEIGHTS_LOCAL      = LOCAL_ROOT / 'weights/outdoor.ckpt'

SOURCE_DIR           = LOCAL_ROOT / 'source_all'
TARGET_DIR           = LOCAL_ROOT / 'target_all'
RETRIEVAL_OUTPUT_DIR = LOCAL_ROOT / 'retrieval_output'
ASPAN_OUTPUT_DIR     = LOCAL_ROOT / 'aspan_output'
VGGT_OUTPUT_DIR      = LOCAL_ROOT / 'vggt_output'

print('DRIVE_ROOT:', DRIVE_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)

In [ ]:
# Run-control flags
RESET_LOCAL_ROOT = False  # WARNING: wipes LOCAL_ROOT and all outputs

RUN_ASPAN        = True   # False = skip (resume from existing outputs)
RUN_VGGT         = True
RUN_POSE_SCORING = True

MAX_PAIRS = None  # set to e.g. 5 for a quick smoke test; None = full run

print('Run-control set.')

---
## Section C — Staging
*Run when resetting or pulling updated files from Drive.  
Cells are idempotent — re-running skips already-present files.*

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def copy_file(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        raise FileNotFoundError(src)
    ensure_dir(dst.parent)
    shutil.copy2(src, dst)
    print(f'Copied {src.name} -> {dst}')
    return dst


def unzip_archive(zip_path, output_dir):
    zip_path, output_dir = Path(zip_path), ensure_dir(output_dir)
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)
    print(f'Unzipping {zip_path.name} -> {output_dir}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(output_dir)
    return output_dir


def list_tree_summary(path, max_items=20):
    path = Path(path)
    print(f'\nTree: {path}')
    if not path.exists():
        print('  [missing]')
        return
    files = sorted(p for p in path.rglob('*') if p.is_file())
    print(f'  files: {len(files)}')
    for p in files[:max_items]:
        print(' ', p.relative_to(path))
    if len(files) > max_items:
        print(f'  ... {len(files) - max_items} more')


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open() as f:
        return sum(1 for line in f if line.strip())


print('Utilities defined.')

In [ ]:
import shutil

# 1. Optional reset
if RESET_LOCAL_ROOT and LOCAL_ROOT.exists():
    print(f'WARNING: Wiping {LOCAL_ROOT} ...')
    shutil.rmtree(LOCAL_ROOT)

# 2. Create output dirs
for d in [LOCAL_ROOT, WEIGHTS_DIR, SOURCE_DIR, TARGET_DIR,
          RETRIEVAL_OUTPUT_DIR, ASPAN_OUTPUT_DIR, VGGT_OUTPUT_DIR]:
    ensure_dir(d)

# 3. Copy pipeline scripts from Drive
for src in [RETRIEVE_SCRIPT_DRIVE, GEOFILTER_SCRIPT_DRIVE,
            VGGTSIGNALS_SCRIPT_DRIVE, POSESCORING_SCRIPT_DRIVE, MODEL_DRIVE]:
    copy_file(src, LOCAL_ROOT / src.name)

# 4. Copy + unzip archives
copy_file(SOURCE_ZIP, LOCAL_ROOT / SOURCE_ZIP.name)
copy_file(TARGET_ZIP, LOCAL_ROOT / TARGET_ZIP.name)
unzip_archive(LOCAL_ROOT / SOURCE_ZIP.name, SOURCE_DIR)
unzip_archive(LOCAL_ROOT / TARGET_ZIP.name, TARGET_DIR)

# 5. Copy checkpoints
for src, dst in [
    (DINO_CHECKPOINT_DRIVE, MATCH_WEIGHTS),
    (DINO_V3_CHECKPOINT_DRIVE, DINO_V3_CHECKPOINT_LOCAL),
    (ASPAN_WEIGHTS_DRIVE, ASPAN_WEIGHTS_LOCAL),
]:
    if src.exists():
        copy_file(src, dst)
    else:
        print(f'WARNING: not found: {src}')

if RUN_VGGT:
    copy_file(VGGT_CHECKPOINT_DRIVE, VGGT_CHECKPOINT_LOCAL)

print('Staging complete.')

In [ ]:
import importlib, sys

if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))
print('sys.path ready.')

---
## Section D — Pipeline
*Each step is independently re-runnable given prior stages produced their outputs.  
Scripts run with default thresholds unless you uncomment the override lines.*

In [ ]:
# Step 1 — Retrieval (retrieve.py)
import os, importlib

import retrieve
importlib.reload(retrieve)
os.chdir(LOCAL_ROOT)

retrieve.main([
    '--weights',          str(MATCH_WEIGHTS),
    '--model-definition', str(LOCAL_ROOT / MODEL_DRIVE.name),
    '--model-class',      MODEL_CLASS,
    '--source',           str(SOURCE_DIR),
    '--target',           str(TARGET_DIR),
    '--output-dir',       str(RETRIEVAL_OUTPUT_DIR),
    # Overrides (uncomment to deviate from defaults):
    # '--topx', '15',         # default: 15
    # '--chunk-size', '256',
    *(['--max-pairs', str(MAX_PAIRS)] if MAX_PAIRS else []),
])

for name in ['retrieval_manifest.jsonl', 'feature_cache.npz']:
    p = RETRIEVAL_OUTPUT_DIR / name
    suffix = f' rows={count_jsonl(p)}' if p.suffix == '.jsonl' else ''
    print(f'{name}: exists={p.exists()}{suffix}')

In [ ]:
# Step 2 — Geometric verification (geometry_filter.py)
import os, importlib

if not RUN_ASPAN:
    print('RUN_ASPAN=False; skipping.')
else:
    import geometry_filter as aspanfilter
    importlib.reload(aspanfilter)
    os.chdir(LOCAL_ROOT)

    aspanfilter.main([
        '--input-manifest', str(RETRIEVAL_OUTPUT_DIR / 'retrieval_manifest.jsonl'),
        '--output-dir',     str(ASPAN_OUTPUT_DIR),
        '--aspanpath',      str(ASPAN_ROOT),
        '--weights_path',   str(ASPAN_WEIGHTS_LOCAL),
        '--config_path',    str(ASPAN_CONFIG),
        '--resume',
        # Overrides:
        # '--breakpoint-value', '50',   # default: 50
        # '--long_dim', '1024',
        *(['--max-pairs', str(MAX_PAIRS)] if MAX_PAIRS else []),
    ])

    for name in ['aspan_all_manifest.jsonl', 'vggt_candidates_manifest.jsonl']:
        p = ASPAN_OUTPUT_DIR / name
        print(f'{name}: exists={p.exists()} rows={count_jsonl(p)}')

In [ ]:
# Step 3 — VGGT-Omega signal generation (vggt_signals.py)
import os, importlib

if not RUN_VGGT:
    print('RUN_VGGT=False; skipping.')
else:
    import vggt_signals
    importlib.reload(vggt_signals)
    os.chdir(LOCAL_ROOT)

    vggt_signals.main([
        '--input-manifest', str(ASPAN_OUTPUT_DIR / 'vggt_candidates_manifest.jsonl'),
        '--output-dir',     str(VGGT_OUTPUT_DIR),
        '--checkpoint',     str(VGGT_CHECKPOINT_LOCAL),
        '--resume',
        # Overrides:
        # '--global-sim-threshold',      '0.90',
        # '--pose-component-threshold',  '3.0',
        # '--pose-score-mode',            'component',
        # '--aspan-prepass-mode',         'lenient',
        # '--max-res', '384',
        *(['--max-pairs', str(MAX_PAIRS)] if MAX_PAIRS else []),
    ])

    for name in ['vggt_judged_manifest.jsonl', 'true_matches_manifest.jsonl', 'vggt_judge_summary.json']:
        p = VGGT_OUTPUT_DIR / name
        rows = count_jsonl(p) if p.suffix == '.jsonl' else 'n/a'
        print(f'{name}: exists={p.exists()} rows={rows}')

In [ ]:
# Step 4 — Decision layer (pose_scoring.py)
import importlib

if not RUN_POSE_SCORING:
    print('RUN_POSE_SCORING=False; skipping.')
else:
    import pose_scoring
    importlib.reload(pose_scoring)

    pose_scoring.main([
        '--input-manifest', str(VGGT_OUTPUT_DIR / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(VGGT_OUTPUT_DIR),
        # Overrides (uncomment to deviate from paper defaults):
        # '--inlier-ratio-threshold',    '0.65',   # default: 0.65
        # '--pose-component-threshold',  '2.13',   # default: 2.13
        # '--pose-component-threshold',  '0',      # disable pose gate (ablation B2)
        # '--pose-components', 'rotation_xy',      # ablation B5
        # '--pose-components', 'fov_z',            # ablation B6
    ])

    for name in ['pose_scored_manifest.jsonl', 'pose_scoring_summary.json']:
        p = VGGT_OUTPUT_DIR / name
        rows = count_jsonl(p) if p.suffix == '.jsonl' else 'n/a'
        print(f'{name}: exists={p.exists()} rows={rows}')

---
## Section E — Inspect
*Run ad-hoc at any point to audit pipeline outputs.*

In [ ]:
import json

outputs = {
    'retrieval_manifest':       RETRIEVAL_OUTPUT_DIR / 'retrieval_manifest.jsonl',
    'feature_cache':            RETRIEVAL_OUTPUT_DIR / 'feature_cache.npz',
    'aspan_all_manifest':       ASPAN_OUTPUT_DIR / 'aspan_all_manifest.jsonl',
    'vggt_candidates_manifest': ASPAN_OUTPUT_DIR / 'vggt_candidates_manifest.jsonl',
    'vggt_judged_manifest':     VGGT_OUTPUT_DIR / 'vggt_judged_manifest.jsonl',
    'true_matches_manifest':    VGGT_OUTPUT_DIR / 'true_matches_manifest.jsonl',
    'vggt_judge_summary':       VGGT_OUTPUT_DIR / 'vggt_judge_summary.json',
    'pose_scored_manifest':     VGGT_OUTPUT_DIR / 'pose_scored_manifest.jsonl',
    'pose_scoring_summary':     VGGT_OUTPUT_DIR / 'pose_scoring_summary.json',
}

print('=== Output audit ===')
for name, path in outputs.items():
    rows = count_jsonl(path) if path.suffix == '.jsonl' else 'n/a'
    print(f'  {name}: exists={path.exists()} rows={rows}')

print()
for name in ['retrieval_manifest', 'aspan_all_manifest', 'vggt_judged_manifest', 'pose_scored_manifest']:
    path = outputs[name]
    if path.exists():
        print(f'--- First row: {name} ---')
        with path.open() as f:
            for line in f:
                if line.strip():
                    print(json.dumps(json.loads(line), indent=2)[:1500])
                    break